# M12 — Improve Weak Trees with Ensembles

Start with a deliberately limited tree, then compare bootstrap resampling, bagging, random forests, and boosting on one fixed split. Every experiment begins with a **Prediction before action**. Record your own prediction in a separate learner log before running the next code cell; this source notebook intentionally contains no learner answers.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

SEED = 1200
DATA_PATH = Path('datasets/M12/ensemble_fixture.csv')
if not DATA_PATH.is_file():
    DATA_PATH = Path.cwd().parent / 'datasets/M12/ensemble_fixture.csv'
if not DATA_PATH.is_file():
    raise FileNotFoundError('Run this notebook from the LearningOS-AI repository root.')

data = pd.read_csv(DATA_PATH)
FEATURES = ['x1', 'x2', 'linear_mix', 'periodic', 'noise1', 'noise2']
X = data[FEATURES]
y = data['target']
print({'rows': len(data), 'features': FEATURES, 'class_counts': y.value_counts().sort_index().to_dict(), 'sklearn': sklearn.__version__})

## Bias/variance intuition and controls

A shallow tree may have high approximation bias; an unconstrained tree may react strongly to the particular training records. Disagreement across bootstrap fits is an observable **sample-sensitivity proxy**, not a formal bias/variance decomposition. All comparisons below hold the seeded stratified split and balanced-accuracy metric fixed.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

def metric_row(name, model):
    train_prediction = model.predict(X_train)
    test_prediction = model.predict(X_test)
    train_balanced = balanced_accuracy_score(y_train, train_prediction)
    test_balanced = balanced_accuracy_score(y_test, test_prediction)
    return {
        'model': name,
        'train_balanced_accuracy': train_balanced,
        'test_balanced_accuracy': test_balanced,
        'test_accuracy': accuracy_score(y_test, test_prediction),
        'generalization_gap': train_balanced - test_balanced,
    }

print({'train_rows': len(X_train), 'test_rows': len(X_test), 'split_seed': SEED})

## Experiment 1 — Limited tree baseline

**Prediction before action:** Will a depth-2 tree underfit, overfit, or balance this non-linear fixture? Predict the train/test direction and the likely gap before fitting.

In [ ]:
limited_tree = DecisionTreeClassifier(max_depth=2, random_state=SEED)
limited_tree.fit(X_train, y_train)
baseline_result = pd.DataFrame([metric_row('limited tree (depth=2)', limited_tree)])
baseline_result.round(3)

## Experiment 2 — Bootstrap variation

**Prediction before action:** Which held-out cases should show the most **prediction variation** across resampled depth-4 trees—confident interior cases or cases near an uncertain boundary? State why.

In [ ]:
def fit_resampled_trees(n_resamples=40, max_depth=4, seed=SEED + 10):
    rng = np.random.default_rng(seed)
    predictions = []
    unique_counts = []
    for sample_number in range(n_resamples):
        sample_indices = rng.integers(0, len(X_train), size=len(X_train))
        tree = DecisionTreeClassifier(max_depth=max_depth, random_state=seed + sample_number)
        tree.fit(X_train.iloc[sample_indices], y_train.iloc[sample_indices])
        predictions.append(tree.predict(X_test))
        unique_counts.append(len(np.unique(sample_indices)))
    return np.asarray(predictions), np.asarray(unique_counts)

bootstrap_predictions, bootstrap_unique_counts = fit_resampled_trees()
positive_vote_rate = bootstrap_predictions.mean(axis=0)
bootstrap_disagreement = 2 * np.minimum(positive_vote_rate, 1 - positive_vote_rate)
variation_frame = X_test[['x1', 'x2']].copy()
variation_frame['actual'] = y_test.to_numpy()
variation_frame['positive_vote_rate'] = positive_vote_rate
variation_frame['disagreement'] = bootstrap_disagreement
print({'mean_unique_rows_per_bootstrap': bootstrap_unique_counts.mean(), 'mean_disagreement': bootstrap_disagreement.mean(), 'unstable_cases': int((bootstrap_disagreement > 0.25).sum())})

In [ ]:
display(variation_frame.sort_values('disagreement', ascending=False).head(10).round(3))
fig, ax = plt.subplots(figsize=(6, 4))
points = ax.scatter(variation_frame['x1'], variation_frame['x2'], c=variation_frame['disagreement'], cmap='magma', vmin=0, vmax=1)
ax.set(title='Bootstrap tree disagreement on held-out cases', xlabel='x1', ylabel='x2')
fig.colorbar(points, ax=ax, label='disagreement (0=stable, 1=split vote)')
plt.show()

## Experiment 3 — Tree vs bagging/RF

Bagging fits trees on bootstrap samples and averages their votes. A random forest adds feature randomness at splits to reduce correlation among trees. **Prediction before action:** Rank a depth-4 tree, bagging, and random forest on held-out balanced accuracy and generalization gap.

In [ ]:
comparison_tree = DecisionTreeClassifier(max_depth=4, random_state=SEED + 20).fit(X_train, y_train)
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=4, random_state=SEED + 21),
    n_estimators=75, bootstrap=True, random_state=SEED + 22, n_jobs=1,
).fit(X_train, y_train)
random_forest_model = RandomForestClassifier(
    n_estimators=75, max_depth=4, max_features='sqrt', bootstrap=True,
    random_state=SEED + 23, n_jobs=1,
).fit(X_train, y_train)

In [ ]:
comparison_results = pd.DataFrame([
    *baseline_result.to_dict('records'),
    metric_row('tree (depth=4)', comparison_tree),
    metric_row('bagging (75 x depth=4)', bagging_model),
    metric_row('random forest (75 x depth=4)', random_forest_model),
])
comparison_results.round(3)

## Experiment 4 — Bagging vs boosting and Sequential correction

Boosting is not bootstrap voting. Each shallow stage is added after the current ensemble and attempts to reduce its loss. **Prediction before action:** Will 75 depth-1 boosting stages beat 75 bagged depth-4 trees here? Predict whether every stage will correct at least one class label.

In [ ]:
boosting_model = GradientBoostingClassifier(
    n_estimators=75, learning_rate=0.08, max_depth=1, random_state=SEED + 30
).fit(X_train, y_train)
staged_predictions = list(boosting_model.staged_predict(X_test))
transition_rows = []
truth = y_test.to_numpy()
for stage_index in range(1, len(staged_predictions)):
    previous_correct = staged_predictions[stage_index - 1] == truth
    current_correct = staged_predictions[stage_index] == truth
    transition_rows.append({
        'stage': stage_index + 1,
        'corrected_cases': int((~previous_correct & current_correct).sum()),
        'newly_wrong_cases': int((previous_correct & ~current_correct).sum()),
        'balanced_accuracy': balanced_accuracy_score(truth, staged_predictions[stage_index]),
    })
stage_transitions = pd.DataFrame(transition_rows)
display(stage_transitions.loc[(stage_transitions['corrected_cases'] + stage_transitions['newly_wrong_cases']) > 0].head(12).round(3))

In [ ]:
bagging_boosting_results = pd.DataFrame([
    metric_row('bagging (75 x depth=4)', bagging_model),
    metric_row('gradient boosting (75 x depth=1)', boosting_model),
])
bagging_boosting_results.round(3)

## Experiment 5 — Number of estimators

**Prediction before action:** For each ensemble, predict where gains will flatten. Is test performance required to improve monotonically when `n_estimators` increases?

In [ ]:
ESTIMATOR_COUNTS = [1, 5, 10, 25, 50, 100, 200]

def run_size_sweep():
    rows = []
    for count in ESTIMATOR_COUNTS:
        models = {
            'bagging': BaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=4, random_state=SEED + 40),
                n_estimators=count, random_state=SEED + 41, n_jobs=1,
            ),
            'random_forest': RandomForestClassifier(
                n_estimators=count, max_depth=4, max_features='sqrt',
                random_state=SEED + 42, n_jobs=1,
            ),
            'gradient_boosting': GradientBoostingClassifier(
                n_estimators=count, learning_rate=0.08, max_depth=1, random_state=SEED + 43,
            ),
        }
        for name, model in models.items():
            started = perf_counter()
            model.fit(X_train, y_train)
            result = metric_row(name, model)
            result.update({'n_estimators': count, 'fit_seconds': perf_counter() - started})
            rows.append(result)
    return pd.DataFrame(rows)


In [ ]:
size_results = run_size_sweep()
display(size_results.round(3))
fig, ax = plt.subplots(figsize=(7, 4))
for name, group in size_results.groupby('model'):
    ax.plot(group['n_estimators'], group['test_balanced_accuracy'], marker='o', label=name)
ax.set(xscale='log', xlabel='number of estimators (log scale)', ylabel='test balanced accuracy', title='Diminishing returns are model-dependent')
ax.legend()
plt.show()

## Experiment 6 — Tree complexity and depth

**Prediction before action:** Predict how depth 1, 2, 4, and unlimited will change training score, held-out score, and the generalization gap for one tree, bagging, and random forest. For boosting, compare depth 1, 2, and 4.

In [ ]:
TREE_DEPTHS = [1, 2, 4, None]

def run_depth_sweep():
    rows = []
    for depth in TREE_DEPTHS:
        models = {
            'tree': DecisionTreeClassifier(max_depth=depth, random_state=SEED + 50),
            'bagging': BaggingClassifier(
                estimator=DecisionTreeClassifier(max_depth=depth, random_state=SEED + 51),
                n_estimators=50, random_state=SEED + 52, n_jobs=1,
            ),
            'random_forest': RandomForestClassifier(
                n_estimators=50, max_depth=depth, max_features='sqrt',
                random_state=SEED + 53, n_jobs=1,
            ),
        }
        if depth is not None:
            models['gradient_boosting'] = GradientBoostingClassifier(
                n_estimators=50, learning_rate=0.08, max_depth=depth, random_state=SEED + 54
            )
        for name, model in models.items():
            model.fit(X_train, y_train)
            result = metric_row(name, model)
            result['depth'] = 'unlimited' if depth is None else str(depth)
            rows.append(result)
    return pd.DataFrame(rows)


In [ ]:
depth_results = run_depth_sweep()
display(depth_results.round(3))
depth_order = {'1': 1, '2': 2, '4': 4, 'unlimited': 6}
fig, ax = plt.subplots(figsize=(7, 4))
for name, group in depth_results.groupby('model'):
    ordered = group.assign(order=group['depth'].map(depth_order)).sort_values('order')
    ax.plot(ordered['depth'], ordered['generalization_gap'], marker='o', label=name)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(xlabel='base-tree depth', ylabel='train − test balanced accuracy', title='Depth changes capacity and generalization gap')
ax.legend()
plt.show()

## Experiment 7 — Controlled failure: excess complexity and “more trees always fixes it”

The fault is a corrupted training signal. **Prediction before action:** Can a 200-tree forest discover which labels were flipped? Predict the clean-test curve and the corrupted-train score before running.

In [ ]:
CORRUPTION_RATE = 0.28
corruption_rng = np.random.default_rng(SEED + 60)
corrupted_y_train = y_train.copy()
corrupted_positions = corruption_rng.choice(
    len(corrupted_y_train), size=round(CORRUPTION_RATE * len(corrupted_y_train)), replace=False
)
corrupted_y_train.iloc[corrupted_positions] = 1 - corrupted_y_train.iloc[corrupted_positions]
failure_rows = []
for count in ESTIMATOR_COUNTS:
    model = RandomForestClassifier(
        n_estimators=count, max_depth=None, max_features='sqrt',
        random_state=SEED + 61, n_jobs=1,
    ).fit(X_train, corrupted_y_train)
    failure_rows.append({
        'n_estimators': count,
        'corrupted_train_balanced_accuracy': balanced_accuracy_score(corrupted_y_train, model.predict(X_train)),
        'clean_test_balanced_accuracy': balanced_accuracy_score(y_test, model.predict(X_test)),
    })
failure_results = pd.DataFrame(failure_rows)
failure_results['gap'] = failure_results['corrupted_train_balanced_accuracy'] - failure_results['clean_test_balanced_accuracy']
failure_results.round(3)

In [ ]:
assert bootstrap_predictions.shape == (40, len(X_test))
assert bootstrap_disagreement.max() > 0
assert set(comparison_results['model']) >= {'limited tree (depth=2)', 'tree (depth=4)', 'bagging (75 x depth=4)', 'random forest (75 x depth=4)'}
assert set(size_results['n_estimators']) == set(ESTIMATOR_COUNTS)
assert set(depth_results['depth']) == {'1', '2', '4', 'unlimited'}
assert failure_results['clean_test_balanced_accuracy'].between(0, 1).all()
best_failure_row = failure_results.loc[failure_results['clean_test_balanced_accuracy'].idxmax()]
largest_failure_row = failure_results.iloc[-1]
print({
    'best_corrupted_test_score': round(float(best_failure_row['clean_test_balanced_accuracy']), 3),
    'best_corrupted_tree_count': int(best_failure_row['n_estimators']),
    'score_at_200_trees': round(float(largest_failure_row['clean_test_balanced_accuracy']), 3),
    'diagnosis_prompt': 'Explain why estimator count cannot restore information destroyed by wrong labels.',
})

## Observation → explanation → generalization

For each experiment, add to your separate evidence record: (1) your original prediction, (2) the observed metric or transition, (3) the mechanism you think explains it, (4) what changed in your belief, and (5) what would need validation on real V03 data. Do not turn a single synthetic holdout into a universal claim.

## ADR hand-off

Use `missions/M12/adr_prompt.md` and `templates/ADR.md` to choose the next V03 ensemble trial. Your **ADR** must bound depth and estimator count, state latency and interpretability constraints, separate fixture observations from production inference, and include measurable revisit conditions. Leave status `Proposed` pending review.

## Code reading

Trace the fitted state and prediction path using `missions/M12/code_reading.md`. Pay special attention to what can be parallelized in bagging/RF and what is sequential in boosting.

## No-AI gate

Close this notebook and complete `missions/M12/no_ai_gate.md` without AI assistance. Reconstruct the three ensemble mechanisms and defend a fresh choice under new operational constraints.

## Formal engineering review

Submit the executed evidence, controlled-failure diagnosis, and proposed ADR using `missions/M12/review_brief.md`. A valid review identifies unsupported claims, operational consequences, unresolved uncertainty, and required follow-ups; it is not a rubber stamp.